# Notebook 16 — Stereo Vision Theory

**Part 6: Stereo Vision** | Estimated time: 50 min

---

ArUco gave us 6D pose **if we print a marker**. But what if the object has no marker?  
**Stereo vision** gives us depth from two cameras — no markers required.

```
Methods we've covered:

  solvePnP      → depth from known 3D points + 2D correspondences (needs calibrated scene)
  ArUco pose    → depth from known marker size (needs printed marker)
  Stereo vision → depth from two camera views (needs only two cameras!) ← YOU ARE HERE
```

## Recommended Watch

> Watch this **before** opening the notebook — it covers the full stereo calibration workflow and maps directly to both NB 16 and NB 17.

| Title | Link | Duration |
|---|---|---|
| **Stereo Vision Camera Calibration with OpenCV: How to Calibrate your Camera with Python Script** | [▶ Watch](https://www.youtube.com/watch?v=yKypaVl6qQo) | ~30 min |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-python numpy matplotlib scipy -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

## Learning Objectives

By the end of this notebook you will be able to:

- Explain how binocular disparity creates depth perception
- Define epipolar geometry: epipoles, epipolar lines, essential matrix, fundamental matrix
- Derive and use the depth formula from disparity, baseline, and focal length
- Understand what image rectification does and why it's needed
- Explain the Q matrix and how it converts disparity to 3D
- Describe how block matching computes disparity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
import cv2

print(f'NumPy: {np.__version__}')
print(f'OpenCV: {cv2.__version__}')

## 1. Human Depth Perception — The Analogy

Close one eye. Reach for a nearby object. Hard, right?  
Open both eyes — suddenly depth is obvious.

Your brain computes depth by comparing the **slightly different images** from each eye.  
A nearby object appears at very different positions in each eye.  
A distant object appears at nearly the same position.

This shift is called **disparity**.

```
                    Left eye view:          Right eye view:

Near object:         ●  (far right)          ● (far left)    ← BIG disparity → close
Far object:          ·  (slightly right)     · (slightly left) ← small disparity → far
```

In [ ]:
# Demonstrate how an object's apparent position shifts with distance
# Setup: two cameras separated by baseline B, both facing forward

B  = 0.10  # Baseline: 10 cm between cameras
fx = 600.0 # Focal length in pixels

# Object at various depths Z
Z_values = np.array([0.3, 0.5, 1.0, 2.0, 5.0])  # meters

# For an object on the camera's central axis (X_world = 0):
# Left camera: object appears at x_L = fx * B/2 / Z  (slightly right)
# Right camera: object appears at x_R = -fx * B/2 / Z (slightly left)
# Disparity d = x_L - x_R = fx * B / Z

disparities = fx * B / Z_values

print('=== Disparity vs Distance ===')
print(f'Baseline: {B*100:.0f} cm | fx: {fx:.0f} px')
print()
print(f'{"Depth Z":>10} | {"Disparity d":>12} | {"Pixel shift":>12}')
print('-' * 40)
for Z, d in zip(Z_values, disparities):
    print(f'{Z*100:>8.0f}cm | {d:>10.2f} px | ~{d:.0f} pixels per camera')

fig, ax = plt.subplots(figsize=(9, 5))
Z_plot = np.linspace(0.1, 6, 300)
d_plot = fx * B / Z_plot
ax.plot(Z_plot, d_plot, 'b-', linewidth=2, label=f'fx={fx:.0f}, B={B*100:.0f}cm')
ax.scatter(Z_values, disparities, c='red', s=80, zorder=5, label='Sample points')
ax.set_xlabel('Distance Z (m)', fontsize=12)
ax.set_ylabel('Disparity d (pixels)', fontsize=12)
ax.set_title('Disparity vs Depth — closer = more pixels of shift', fontsize=13)
ax.legend()
ax.set_xlim(0, 6)
ax.set_ylim(0, 250)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. The Depth Formula

From the diagram above, we can derive the central formula of stereo vision:

$$Z = \frac{f \cdot B}{d}$$

Where:
- $Z$ = **depth** (distance from camera to point) — what we want
- $f$ = **focal length** in pixels — from camera calibration
- $B$ = **baseline** in meters — physical distance between camera centers
- $d$ = **disparity** in pixels — how many pixels the point shifts between views

**Intuitions:**
- Large disparity $d$ → small $Z$ → object is **close**
- Small disparity $d$ → large $Z$ → object is **far**
- Larger baseline $B$ → more accurate depth (better separation)
- Shorter focal length → smaller disparity for same depth (harder to measure)

**Minimum measurable depth** (when disparity = image width W):
$$Z_{min} = \frac{f \cdot B}{W}$$

**Maximum reliable depth** (when disparity ≈ 1 pixel):
$$Z_{max} \approx f \cdot B$$

In [ ]:
def disparity_to_depth(disparity_px, focal_length_px, baseline_m):
    """Convert disparity map values to depth in meters."""
    # Avoid division by zero
    safe_disp = np.where(disparity_px > 0, disparity_px, np.inf)
    return (focal_length_px * baseline_m) / safe_disp

def depth_to_disparity(depth_m, focal_length_px, baseline_m):
    """Convert depth in meters to disparity in pixels."""
    return (focal_length_px * baseline_m) / depth_m

# Practical example: our stereo camera setup
fx = 600.0   # pixels
B  = 0.12    # 12 cm baseline (typical USB stereo camera)
W  = 640     # image width

Z_min = fx * B / W
Z_max = fx * B / 1.0  # 1 pixel min disparity

print(f'Stereo setup: fx={fx:.0f}px, B={B*100:.0f}cm, W={W}px')
print(f'Depth range: {Z_min*100:.1f}cm to {Z_max:.1f}m')
print()

# Test some values
test_disparities = np.array([50, 30, 20, 10, 5, 2], dtype=float)
depths = disparity_to_depth(test_disparities, fx, B)
print(f'{"Disparity":>12} | {"Depth":>12}')
print('-' * 30)
for d, Z in zip(test_disparities, depths):
    print(f'{d:>10.0f}px | {Z*100:>10.1f}cm')

## 3. Epipolar Geometry

To match points between the two camera views, we need to limit the **search space**.  
Without constraints, we'd have to search the entire image for each point — $O(N^2)$.

**Epipolar geometry** gives us a critical constraint:

> If a point $p$ appears at pixel $(u, v)$ in the left image,  
> then the **same 3D point** must appear somewhere on a specific **line** in the right image.
> That line is called the **epipolar line**.

```
Left camera                    Right camera
    │                              │
    │   Point P in 3D            │
    │       /                     │
    │      /                      │    ←─── Epipolar line
    │     /  ← epipolar plane     │         (where P must appear)
    │    /                        │
    │   / p_left                  │
    │  ●                          │  ──────────────●──────── (search only here)
    │                              │
  Left optical center           Right optical center
  (left epipole of right cam)   (right epipole of left cam)
```

This reduces stereo matching from 2D search to **1D search along a line** — much faster and more reliable.

### The Epipolar Constraint Math

For calibrated cameras (we know K), the **Essential Matrix E** encodes epipolar geometry:

$$\mathbf{x}_R^T E \mathbf{x}_L = 0$$

Where:
- $\mathbf{x}_L$ = normalized point in left image (K removed)
- $\mathbf{x}_R$ = normalized point in right image (K removed)
- $E = [t]_\times R$ — encodes rotation R and translation t between cameras

For uncalibrated cameras, the **Fundamental Matrix F** works directly with pixel coordinates:

$$\mathbf{p}_R^T F \mathbf{p}_L = 0$$

Where $F = K_R^{-T} E K_L^{-1}$

In [ ]:
# Visualize the epipolar constraint with a diagram
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Left and right image (simulated)
left_img  = np.ones((400, 640, 3), dtype=np.uint8) * 220
right_img = np.ones((400, 640, 3), dtype=np.uint8) * 220

# Add some fake scene content
cv2.rectangle(left_img,  (100, 80),  (200, 180), (180, 180, 180), -1)
cv2.rectangle(left_img,  (350, 120), (500, 300), (160, 160, 200), -1)
cv2.rectangle(right_img, (80,  80),  (180, 180), (180, 180, 180), -1)
cv2.rectangle(right_img, (310, 120), (460, 300), (160, 160, 200), -1)

# Point in left image
p_left = (150, 130)
cv2.circle(left_img, p_left, 10, (255, 0, 0), -1)
cv2.putText(left_img, 'p_L (query point)', (p_left[0]+15, p_left[1]),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

# Epipolar line in right image at same v-coordinate (after rectification)
v_epi = p_left[1]  # same row after rectification
cv2.line(right_img, (0, v_epi), (640, v_epi), (0, 0, 255), 2)
cv2.putText(right_img, f'Epipolar line (row {v_epi})',
            (10, v_epi - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

# Matching point in right image
p_right = (120, v_epi)  # same row, shifted left by disparity
cv2.circle(right_img, p_right, 10, (0, 200, 0), -1)
cv2.putText(right_img, f'p_R (match, d={p_left[0]-p_right[0]}px)',
            (p_right[0]+15, p_right[1]-15),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)

axes[0].imshow(cv2.cvtColor(left_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Left camera — query point (blue)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(right_img, cv2.COLOR_BGR2RGB))
axes[1].set_title('Right camera — epipolar line (red) + match (green)', fontsize=12)
axes[1].axis('off')

plt.suptitle('Epipolar Constraint: 2D search → 1D search along epipolar line',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key insight: After rectification, epipolar lines become horizontal rows.')
print('This makes stereo matching simply: scan along the SAME ROW in both images.')

## 4. Why Rectification Matters

In theory, epipolar lines are straight but not necessarily horizontal.  
They become horizontal only if the cameras are perfectly aligned side-by-side.

In practice, two cameras mounted on a rig are **never perfectly aligned**.  
**Stereo rectification** warps both images so that:

1. Epipolar lines become **horizontal scan lines**
2. Corresponding row numbers are the **same** in both images
3. Disparity is only in the **X direction** (no vertical component)

After rectification, stereo matching is trivially reduced to:

```python
for each row r:
    for each column c in left_image[r]:
        search for match in right_image[r][0:c]  # same row, to the left
```

In [ ]:
# Visualize what rectification does to the image geometry
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

def make_fake_scene(skew_angle=0, flip=False):
    img = np.ones((400, 640, 3), dtype=np.uint8) * 200
    colors = [(150,180,200), (200,150,180), (180,200,150)]
    for i, (box, c) in enumerate([
            ((80,60,200,200), colors[0]),
            ((300,80,500,350), colors[1]),
            ((550,100,620,300), colors[2])]):
        x0,y0,x1,y1 = box
        cv2.rectangle(img, (x0,y0),(x1,y1), c, -1)
        cv2.rectangle(img, (x0,y0),(x1,y1), (100,100,100), 1)
    if skew_angle != 0:
        h,w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2, h//2), skew_angle, 1.0)
        img = cv2.warpAffine(img, M, (w,h), borderValue=(200,200,200))
    return img

left_raw  = make_fake_scene(skew_angle=2.5)
right_raw = make_fake_scene(skew_angle=-1.5)
left_rect  = make_fake_scene()
right_rect = make_fake_scene()

# Draw epipolar lines (non-horizontal on raw, horizontal on rectified)
def draw_epipolar_lines(img, horizontal=True, color=(255,50,50)):
    out = img.copy()
    h, w = out.shape[:2]
    if horizontal:
        for y in range(50, h, 60):
            cv2.line(out, (0,y), (w,y), color, 1)
    else:
        for i, (x,y) in enumerate([(0,400),(0,300),(0,200)]):
            slope_angle = -15 + i*10
            xe = w
            ye = int(y - xe * np.tan(np.radians(slope_angle)))
            cv2.line(out, (x,y), (xe, ye), color, 1)
    return out

axes[0,0].imshow(cv2.cvtColor(draw_epipolar_lines(left_raw, False), cv2.COLOR_BGR2RGB))
axes[0,0].set_title('Left RAW — epipolar lines (tilted)', color='red')
axes[0,0].axis('off')

axes[0,1].imshow(cv2.cvtColor(draw_epipolar_lines(right_raw, False), cv2.COLOR_BGR2RGB))
axes[0,1].set_title('Right RAW — epipolar lines (tilted, different angle)', color='red')
axes[0,1].axis('off')

axes[1,0].imshow(cv2.cvtColor(draw_epipolar_lines(left_rect, True), cv2.COLOR_BGR2RGB))
axes[1,0].set_title('Left RECTIFIED — epipolar lines (horizontal ✓)', color='green')
axes[1,0].axis('off')

axes[1,1].imshow(cv2.cvtColor(draw_epipolar_lines(right_rect, True), cv2.COLOR_BGR2RGB))
axes[1,1].set_title('Right RECTIFIED — same horizontal scan lines ✓', color='green')
axes[1,1].axis('off')

plt.suptitle('Rectification: warps images so epipolar lines become horizontal rows',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Disparity Maps

A **disparity map** is an image where each pixel value = the disparity (pixel shift) at that location.  
Bright pixels = large disparity = **close** objects.  
Dark pixels = small disparity = **far** objects.

### Block Matching Algorithm

The most common method for computing disparity:

```
For each pixel (u, v) in left image:
    1. Extract a small window W×W around (u, v)        ← the "block"
    2. Scan along row v in right image from u-maxDisp to u
    3. Find the position u_R that minimizes SSD (sum of squared differences)
    4. disparity(u, v) = u - u_R
```

OpenCV implements two variants:
- `StereoBM` — fast but rough, good for real-time
- `StereoSGBM` — Semi-Global Block Matching, slower but much better quality

In [ ]:
# Simulate a disparity map from a synthetic stereo pair
def create_synthetic_stereo(width=640, height=480):
    """
    Create a synthetic rectified stereo pair with known disparities.
    Three planes at different depths: background, mid, foreground.
    """
    np.random.seed(42)
    
    # Create depth map (known ground truth)
    depth_map = np.ones((height, width)) * 3.0  # 3m background
    
    # Mid-ground object at 1.5m
    depth_map[100:350, 150:450] = 1.5
    
    # Foreground object at 0.5m
    depth_map[150:280, 220:380] = 0.5
    
    # Convert depth to disparity (d = f*B/Z)
    f  = 600.0  # focal length
    B  = 0.12   # 12cm baseline
    disp_map = (f * B / depth_map).astype(np.float32)
    
    # Create left image (textured to make it interesting)
    left_img = np.random.randint(100, 200, (height, width, 3), dtype=np.uint8)
    # Add some texture
    for y in range(0, height, 30):
        for x in range(0, width, 30):
            color = np.random.randint(50, 200, 3).tolist()
            cv2.rectangle(left_img, (x,y), (x+28,y+28), color, -1)
    
    # Foreground object — red box
    cv2.rectangle(left_img, (220,150), (380,280), (60, 80, 200), -1)
    cv2.putText(left_img, 'Object', (235, 220),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
    
    # Mid object — blue box
    cv2.rectangle(left_img, (150,100), (450,350), (180, 120, 60), -1)
    cv2.rectangle(left_img, (220,150), (380,280), (60, 80, 200), -1)
    cv2.putText(left_img, 'Object', (235, 220),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
    
    return left_img, disp_map, depth_map

left_img, disp_map, depth_map = create_synthetic_stereo()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(cv2.cvtColor(left_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Left image', fontsize=12)
axes[0].axis('off')

im1 = axes[1].imshow(disp_map, cmap='hot')
axes[1].set_title('Disparity map\n(bright = large disparity = close)', fontsize=12)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, label='disparity (px)')

im2 = axes[2].imshow(depth_map, cmap='viridis_r')
axes[2].set_title('Depth map (ground truth)\n(bright = close)', fontsize=12)
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, label='depth (m)')

plt.suptitle('Synthetic stereo pair — disparity and depth', fontsize=13)
plt.tight_layout()
plt.show()

## 6. The Q Matrix — Disparity to 3D

Given a disparity map, we want to reconstruct **3D point positions** for every pixel.

OpenCV provides `cv2.reprojectImageTo3D(disparity, Q)` which uses the **Q matrix**:

$$\begin{bmatrix} X \\ Y \\ Z \\ W \end{bmatrix} = Q \begin{bmatrix} u \\ v \\ d \\ 1 \end{bmatrix}$$

The Q matrix structure:

$$Q = \begin{bmatrix} 1 & 0 & 0 & -c_x \\ 0 & 1 & 0 & -c_y \\ 0 & 0 & 0 & f_x \\ 0 & 0 & -1/B & (c_x - c_x') / B \end{bmatrix}$$

Where:
- $(c_x, c_y)$ = principal point of left camera
- $c_x'$ = principal point x of right camera (for x-offset between cameras)
- $f_x$ = focal length
- $B$ = baseline

The 3D point is then $(X/W, Y/W, Z/W)$.  

> **Good news:** `cv2.stereoRectify()` computes Q for you automatically!

In [ ]:
# Build Q matrix manually to understand it
def build_Q(fx, cx, cy, cx_prime, baseline_m):
    """
    Build the Q matrix for stereo reprojection.
    
    Args:
        fx:        focal length (pixels)
        cx, cy:    left camera principal point (pixels)
        cx_prime:  right camera principal point x (pixels)
        baseline_m: baseline in meters
    """
    Q = np.array([
        [1, 0,  0,         -cx],
        [0, 1,  0,         -cy],
        [0, 0,  0,          fx],
        [0, 0, -1/baseline_m, (cx - cx_prime) / baseline_m]
    ], dtype=np.float64)
    return Q

# Example stereo camera parameters
fx_ex  = 600.0
cx_ex  = 320.0
cy_ex  = 240.0
cx_r   = 318.0  # right camera principal point (slightly different)
B_ex   = 0.12

Q = build_Q(fx_ex, cx_ex, cy_ex, cx_r, B_ex)
print('Q matrix:')
print(Q)

# Test: reproject a point with known disparity
u, v, d = 320, 240, 30.0  # center pixel with 30px disparity
uvd1 = np.array([u, v, d, 1.0])
XYZW = Q @ uvd1
X, Y, Z = XYZW[:3] / XYZW[3]

expected_Z = fx_ex * B_ex / d
print(f'\nTest point: u={u}, v={v}, d={d}px')
print(f'Q reprojection: X={X:.3f}m, Y={Y:.3f}m, Z={Z:.3f}m')
print(f'Expected Z (f*B/d): {expected_Z:.3f}m')
print(f'Match: {abs(Z - expected_Z) < 0.001}')

In [ ]:
# Use Q to reconstruct 3D from our synthetic disparity map
def disparity_to_pointcloud(disp_map, Q, min_disp=1.0):
    """Convert a disparity map to a 3D point cloud using Q matrix."""
    h, w = disp_map.shape
    points = []
    
    # Sample every 10th pixel for speed
    for v in range(0, h, 10):
        for u in range(0, w, 10):
            d = disp_map[v, u]
            if d < min_disp:
                continue
            uvd1 = np.array([u, v, d, 1.0])
            XYZW = Q @ uvd1
            X, Y, Z = XYZW[:3] / XYZW[3]
            if 0 < Z < 10:  # reasonable depth range
                points.append([X, Y, Z])
    
    return np.array(points)

points_3d = disparity_to_pointcloud(disp_map, Q, min_disp=5.0)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Color by depth
z_vals = points_3d[:, 2]
sc = ax.scatter(points_3d[:, 0], points_3d[:, 2],
                -points_3d[:, 1],
                c=z_vals, cmap='viridis_r',
                s=1, alpha=0.6)
plt.colorbar(sc, ax=ax, label='Depth Z (m)', shrink=0.6)
ax.set_xlabel('X (m)')
ax.set_ylabel('Z depth (m)')
ax.set_zlabel('Y (m, inverted)')
ax.set_title('3D Point Cloud from Disparity Map', fontsize=13)
ax.view_init(elev=25, azim=-60)
plt.tight_layout()
plt.show()

print(f'Total 3D points: {len(points_3d)}')
print(f'Depth range: {z_vals.min():.2f}m to {z_vals.max():.2f}m')

## 7. StereoBM and StereoSGBM

In practice, you call OpenCV's stereo matching algorithms rather than implementing block matching yourself.

| Feature | StereoBM | StereoSGBM |
|---|---|---|
| Speed | Fast (real-time @ 640×480) | Slower (5-10 FPS @ 640×480) |
| Quality | Rough, noisy | Much smoother |
| Memory | Low | Moderate |
| Best for | Fast robotics, large features | Accuracy needed, rich texture |
| Key params | `numDisparities`, `blockSize` | Same + `P1`, `P2` penalties |

Key parameters:

| Parameter | Meaning | Rule of thumb |
|---|---|---|
| `numDisparities` | Max expected pixel shift (divisible by 16) | `fx * B / Z_min` rounded up to 16x |
| `blockSize` | Matching window size (odd) | 5–15; larger = smoother but less detail |
| `P1` (SGBM) | Penalty for 1-pixel disparity change | `8 * 3 * blockSize²` |
| `P2` (SGBM) | Penalty for larger changes | `32 * 3 * blockSize²` |

In [ ]:
# Demonstrate StereoBM and StereoSGBM parameters
# We'll create a simple rectified stereo pair and compute disparity

def create_rectified_stereo_pair(width=640, height=480, baseline_px=30):
    """
    Create a synthetic ALREADY RECTIFIED stereo pair.
    The right image is just the left image shifted horizontally
    by different amounts in different regions (simulating disparity).
    """
    np.random.seed(7)
    
    # Rich texture (important for block matching!)
    left = np.random.randint(80, 200, (height, width), dtype=np.uint8)
    # Add smooth regions
    left = cv2.GaussianBlur(left, (5,5), 0)
    
    # Add structured content
    cv2.rectangle(left, (150,100),(430,350), 140, -1)
    cv2.rectangle(left, (200,150),(380,300), 80, -1)
    cv2.rectangle(left, (240,180),(340,270), 180, -1)
    # Add noise to avoid zero-gradient regions
    left = cv2.add(left, np.random.randint(-15,15,(height,width),dtype=np.int8).view(np.uint8).reshape(height,width))
    
    # Create right image by shifting pixels left (positive disparity)
    # Background: 10px shift, midground: 25px, foreground: 45px
    right = np.zeros_like(left)
    
    # Background
    d_bg = 10
    right[:, :width-d_bg] = left[:, d_bg:]
    
    # Midground region
    d_mid = 25
    right[100:350, 150:430-d_mid] = left[100:350, 150+d_mid:430]
    
    # Foreground
    d_fg = 45
    right[150:300, 200:340-d_fg] = left[150:300, 200+d_fg:340]
    
    return left, right

left_gray, right_gray = create_rectified_stereo_pair()

# StereoBM
stereo_bm = cv2.StereoBM_create(
    numDisparities=64,  # must be divisible by 16
    blockSize=15        # odd number
)
disp_bm = stereo_bm.compute(left_gray, right_gray).astype(np.float32) / 16.0

# StereoSGBM
bs = 5
stereo_sgbm = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=64,
    blockSize=bs,
    P1=8 * 1 * bs**2,
    P2=32 * 1 * bs**2,
    disp12MaxDiff=1,
    uniquenessRatio=15,
    speckleWindowSize=100,
    speckleRange=32
)
disp_sgbm = stereo_sgbm.compute(left_gray, right_gray).astype(np.float32) / 16.0

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(left_gray, cmap='gray')
axes[0].set_title('Left image (input)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(right_gray, cmap='gray')
axes[1].set_title('Right image (input)', fontsize=11)
axes[1].axis('off')

disp_bm_show = np.clip(disp_bm, 0, 64)
im1 = axes[2].imshow(disp_bm_show, cmap='hot')
axes[2].set_title('StereoBM\n(fast, rougher)', fontsize=11)
axes[2].axis('off')
plt.colorbar(im1, ax=axes[2], fraction=0.046)

disp_sgbm_show = np.clip(disp_sgbm, 0, 64)
im2 = axes[3].imshow(disp_sgbm_show, cmap='hot')
axes[3].set_title('StereoSGBM\n(slower, smoother)', fontsize=11)
axes[3].axis('off')
plt.colorbar(im2, ax=axes[3], fraction=0.046)

plt.suptitle('Stereo matching comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Recap

| Concept | Key Point |
|---|---|
| Disparity | Pixel shift of same 3D point between left and right views |
| Depth formula | $Z = f \cdot B / d$ — closer = bigger disparity |
| Epipolar constraint | Match must lie on epipolar line — reduces search to 1D |
| Rectification | Warps images so epipolar lines are horizontal scan lines |
| Q matrix | Converts (u, v, disparity) → (X, Y, Z) in 3D space |
| StereoBM | Fast block matching — real-time, rougher quality |
| StereoSGBM | Semi-global BM — slower, much smoother disparity maps |
| Texture matters | Flat textureless surfaces fail — stereo needs visual features |

**Next notebook:** How to actually calibrate a stereo camera pair with OpenCV.

---
## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Stereo system design
# ============================================================
# You need to detect pallets at 1-4m range.
# Given: fx=800px, target depth accuracy ±5cm at 2m.
# (a) What minimum baseline B do you need?
# (b) What is the disparity range?
# Use Z = f*B/d → dZ/dd = -f*B/d² to find the required disparity precision.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# fx = 800; Z_target = 2.0; dZ = 0.05  # 5cm accuracy
# # dZ/dd = f*B/d² → B/d = dZ * d / f (tricky without knowing B)
# # Alternative: 1 pixel disparity error at Z=2m should be <= 5cm
# # dZ = Z^2 / (f*B)  → B = Z^2 / (f * dZ)
# B_needed = Z_target**2 / (fx * dZ)
# print(f'Minimum baseline: {B_needed*100:.1f} cm')
# # Disparity range: from Z=1m to Z=4m
# d_max = fx * B_needed / 1.0
# d_min = fx * B_needed / 4.0
# print(f'Disparity range: {d_min:.1f} to {d_max:.1f} pixels')
# print(f'numDisparities should be >= {int(np.ceil(d_max/16)*16)}')

In [ ]:
# ============================================================
# EXERCISE 2: Depth from Q matrix
# ============================================================
# Use the Q matrix built in section 6.
# For pixels at (320, 240), (100, 200), (500, 300):
#   - Compute depth if disparity = 20px
#   - Compute depth if disparity = 5px
# Print a formatted table.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# pixels = [(320, 240), (100, 200), (500, 300)]
# disparities = [20.0, 5.0]
# print(f'{"Pixel":>14} | {"d=20px Z":>12} | {"d=5px Z":>12}')
# print('-'*45)
# for u, v in pixels:
#     depths = []
#     for d in disparities:
#         XYZW = Q @ np.array([u, v, d, 1.0])
#         depths.append(XYZW[2] / XYZW[3])
#     print(f'({u:3d},{v:3d}):    {depths[0]*100:>10.1f}cm  {depths[1]*100:>10.1f}cm')

In [ ]:
# ============================================================
# EXERCISE 3: Depth accuracy vs. baseline trade-off
# ============================================================
# Plot depth accuracy (dZ for 1 pixel disparity error) vs distance Z
# for three baseline values: B = 5cm, 12cm, 25cm.
# Use dZ ≈ Z² / (f * B), with f = 600px.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# f = 600.0
# Z_range = np.linspace(0.3, 5, 200)
# baselines = [0.05, 0.12, 0.25]
# fig, ax = plt.subplots(figsize=(9,5))
# for B in baselines:
#     dZ = Z_range**2 / (f * B)
#     ax.plot(Z_range, dZ*100, label=f'B={B*100:.0f}cm')
# ax.set_xlabel('Distance Z (m)')
# ax.set_ylabel('Depth error per pixel (cm)')
# ax.set_title('Depth accuracy vs baseline — larger baseline = better accuracy')
# ax.legend()
# ax.axhline(5, color='red', linestyle='--', label='5cm threshold')
# ax.set_ylim(0, 40); ax.grid(True, alpha=0.3)
# plt.tight_layout(); plt.show()